# 18 · One model, a model told the QP, or a specialist?

A frame's slice QP says how hard the codec squeezed it, and the client reads it off every slice header
(desktop-streamer, `stream-h264.js`). A blind model has to hedge between sharpening a clean frame and deblocking a
coded one. Is it worth telling the model the QP, or training a model for high QP alone?

Three models with notebook 7's body (24 channels, 2 residual blocks), trained with the shared recipe
(`training.fit`) on notebook 0's set, which spans slice QP 0 to 51 on both of the host's lines:

1. **Blind**: not told the QP. Notebooks 4 and 8 trained it at three seeds, the ruler for luck.
2. **Told the QP**: each patch's slice QP / 51 as a second input (`models.ToldLuma`), at two seeds.
3. **Specialist**: blind, trained and selected only on patches at slice QP 38 and up.

Each is scored on the validation split, overall and by slice QP bin. A separate high-QP model pays only if the
specialist clearly beats the told model there. A told model ships as the blind graph with its biases a straight line
in the QP (notebook 19), so telling costs the client nothing at run time.

Needs: `00_data_preprocess.ipynb`, and the scores of notebooks 1 (bicubic), 4 and 8 (the blind model at three seeds).

In [1]:
"""Settings."""

import os

import numpy as np
import torch

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, training
from helpers.training import DATA_DIR, DEVICE

NOTEBOOK = 18
WIDTH, BLOCKS, STEM = 24, 2, 3   # notebook 7's body
SPECIALIST_QP = (38, 99)         # the slice QP the specialist trains and selects on
TOLD = ("qp",)
COLUMNS = patches.columns(TOLD)
RUN_DIR = training.CHECKPOINT_DIR / "qp_runs"

manifest = training.load_manifest()
SCALE = manifest["scale"]
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]   # notebook 1's scores of the bar
blind = {name: scores for (notebook, name), scores in recorded.items()
         if notebook in (4, 8) and name.startswith("luma 24 + 2 blocks")}   # the blind model at three seeds


def build(told, seed):
    """The body, told or blind; the same seed starts both with the same weights."""
    torch.manual_seed(seed)
    if told:
        return models.ToldLuma(len(COLUMNS), WIDTH, BLOCKS, STEM, scale=SCALE).to(DEVICE)
    return models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE).to(DEVICE)


RUNS = {   # name -> (told, seed, slice QP range, run file)
    "told the QP": (True, 0, None, RUN_DIR / "told_qp_seed0.pt"),
    "told the QP, seed 1": (True, 1, None, RUN_DIR / "told_qp_seed1.pt"),
    "specialist QP 38+": (False, 0, SPECIALIST_QP, RUN_DIR / "specialist_qp38_seed0.pt"),
}

## 1. Train

`training.fit`, the shared recipe. The told model at seeds 0 and 1; the specialist at seed 0, trained and selected on
slice QP 38 and up only. Each run is saved to `checkpoints/qp_runs/` after every epoch.

In [ ]:
runs = {}
for name, (told, seed, qp_range, path) in RUNS.items():
    model = build(told, seed)
    history, best = training.fit(model, path, manifest, told=TOLD if told else None, seed=seed, qp_range=qp_range,
                                 verbose=0)
    runs[name] = model.eval()
    print(f"{name:<22} best epoch {best['epoch']:3} of {len(history):3}, select SSIM-Y {best['score']:.4f}")

## 2. Gain over bicubic

Every model on the whole validation split, then by slice QP bin. The specialist is scored on every bin, but it was
only trained for QP 38 and up.

In [ ]:
scores = {}
for name, model in runs.items():
    if RUNS[name][0]:
        scores[name] = training.val_scores(lambda lr, condition: model(lr, condition[:, COLUMNS]), manifest)
    else:
        scores[name] = training.val_scores(lambda lr, condition: model(lr), manifest)
    training.record(NOTEBOOK, name, scores[name])

blind_mean = {key: np.mean([s[key] for s in blind.values()], axis=0) for key in ("ssim", "y")}
training.show(training.compare({"bicubic": bicubic, **blind, **scores}, manifest, base=bicubic))
print()
training.show(training.gain_by_qp({"blind, mean of 3 seeds": blind_mean, **scores}, manifest, base=bicubic))

### More than luck?

By slice QP bin, in SSIM-Y: the told model's lead over the blind one (the mean of its two seeds against the mean of
the blind model's three), how far apart the blind seeds and the two told seeds land in that bin, and the specialist
against the told model.

In [ ]:
qp = np.load(DATA_DIR / manifest["splits"]["val"]["qp"])
edges = manifest["qp_edges"]
told_names = ["told the QP", "told the QP, seed 1"]
print(f"{'QP':>7}{'patches':>9}{'told - blind':>14}{'blind seeds':>13}{'told seeds':>12}{'specialist - told':>19}")
for low, high in zip(edges[:-1], edges[1:]):
    inside = (qp >= low) & (qp < high)
    mean_in = lambda s: s["ssim"][inside].mean()
    told_mean = np.mean([mean_in(scores[name]) for name in told_names])
    blind_in = [mean_in(s) for s in blind.values()]
    print(f"{f'{low}-{high}':>7}{inside.sum():9}{told_mean - np.mean(blind_in):+14.4f}{np.ptp(blind_in):13.4f}"
          f"{abs(mean_in(scores[told_names[0]]) - mean_in(scores[told_names[1]])):12.4f}"
          f"{mean_in(scores['specialist QP 38+']) - told_mean:+19.4f}")

## 3. What it says

*(written after the run)*